# Hierarchical ZenML pipelines: parent-child orchestration and artifact lineage

I wanted to understand how ZenML handles nested pipelines — where one pipeline calls another — and how artifact lineage gets tracked across that boundary. This notebook documents my experiment with a child training pipeline embedded inside a parent orchestration pipeline.

last_verified: 2026-07-14 · ZenML n/a

## What I'm building

A **child pipeline** that loads data, preprocesses it, and trains a small model. A **parent pipeline** that calls the child, then evaluates the returned model and logs the final accuracy. I want to see:

- How the parent receives artifacts from the child
- Whether ZenML's dashboard shows the parent-child relationship in the DAG
- How I can query the lineage later using the post-execution API

In [ ]:
from zenml import pipeline, step
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

import warnings
warnings.filterwarnings("ignore")

## Child pipeline: data loading, preprocessing, and training

I'll define three steps inside the child pipeline. Each step returns its output so ZenML can persist it as an artifact and pass it downstream.

In [ ]:
@step
def load_data() -> tuple:
    """Load iris and return features + labels."""
    X, y = load_iris(return_X_y=True)
    return X, y


@step
def preprocess(
    X: np.ndarray, y: np.ndarray
) -> tuple:
    """Split into train/test and return four arrays."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    return X_train, X_test, y_train, y_test


@step
def train_model(
    X_train: np.ndarray, y_train: np.ndarray
) -> RandomForestClassifier:
    """Train a small RandomForest and return the fitted model."""
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)
    return model


@pipeline
def child_training_pipeline():
    X, y = load_data()
    X_train, X_test, y_train, y_test = preprocess(X, y)
    model = train_model(X_train, y_train)
    # Return test split + model so the parent can evaluate
    return X_test, y_test, model

## Parent pipeline: orchestrating the child and evaluating output

The parent calls `child_training_pipeline()` directly inside its body. The returned artifacts (test data and model) are unpacked and passed to an evaluation step. This is the pattern I was most curious about — does ZenML wire up the lineage automatically when I call a pipeline like a regular function?

In [ ]:
@step
def evaluate_model(
    model: RandomForestClassifier,
    X_test: np.ndarray,
    y_test: np.ndarray,
) -> float:
    """Evaluate the model on held-out data."""
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Parent evaluation accuracy: {acc:.4f}")
    return acc


@pipeline
def parent_pipeline():
    # Call the child pipeline and unpack its artifacts
    X_test, y_test, model = child_training_pipeline()
    final_accuracy = evaluate_model(model, X_test, y_test)
    return final_accuracy

## Run the parent pipeline and inspect results

I'll execute the parent pipeline with the default local stack. After the run, I'll use the post-execution API to inspect the pipeline DAG and verify that ZenML linked the parent and child runs together.

In [ ]:
if __name__ == "__main__":
    acc = parent_pipeline()
    print(f"Parent pipeline returned accuracy: {acc:.4f}")

## Inspecting lineage through the post-execution API

Now I want to verify that ZenML recorded the relationship. I'll query the metadata store for the most recent pipeline runs and print the DAG structure.

In [ ]:
from zenml.client import Client

client = Client()

# Query recent pipelines from the metadata store
pipelines = client.list_pipelines()

for pipeline_info in pipelines.items:
    print(f"Pipeline: {pipeline_info.name} (ID: {pipeline_info.id})")
    for run in pipeline_info.runs[:3]:
        print(f"  Run {run.id}: status={run.status}")
        for step in run.steps:
            print(f"    Step {step.name}: {step.status}")

## Got stuck on

- **Returning multiple artifacts from the child pipeline:** I first tried returning a dictionary, but unpacking it in the parent was awkward with ZenML's type hints. Switching to a plain tuple worked immediately.
- **Step type hints:** I used `tuple` instead of explicit `tuple[np.ndarray, np.ndarray]` because ZenML's materializer inference was noisy with generic type hints in this version. I'll pin a version and test again later.
- **Pipeline caching:** The child pipeline ran from scratch every time even when I re-ran the parent with the same inputs. I need to look up the caching configuration.
- **Dashboard DAG:** The ZenML dashboard showed the parent and child as separate runs. I expected a single DAG with nested nodes, but they appear as sibling runs linked by artifact URIs. I may need a different visualisation approach.

## What I'd try next

- Look up pipeline caching and see if the child gets skipped on repeat runs
- Use the ZenML dashboard's artifact tab to trace the model artifact from child train step to parent evaluate step
- Replace the local orchestrator with a remote one to see how parent-child DAGs render in a different UI
- Add a third pipeline level (grandparent) to stress-test the lineage depth